# negative-back — faded example 2: complete the double-negate chain

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `negative-back`. Running the beacon reports progress on the `Backprop: negative_back` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: negative_back` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`negative-back`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "negative-back"
DD_SUBTOPIC = "Backprop: negative_back"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Two single-parent backward functions compose by plain function composition. For `y = -(-x)`, the two sign flips cancel, so the leaf gradient equals `grad_out` exactly.

## Faded exercise 2

`negative_back` is given. Complete `chain_double_negate` so the second backward call propagates the gradient from the inner negate to the leaf. Fill in the missing `g_x` step.

**Fill in:** the second negative_back call that produces the leaf gradient g_x

In [ ]:
import torch as t

t.manual_seed(4)

def negative_back(grad_out, out, x):
    return -grad_out

def chain_double_negate(grad_out, x_leaf):
    u = -x_leaf
    y = -u
    g_u = negative_back(grad_out, y, u)
    g_x = None  # TODO: the second negative_back call that produces the leaf gradient g_x
    return g_x

x = t.randn(3)
grad_out = t.randn(3)
print(chain_double_negate(grad_out, x))


def _test():
    t.manual_seed(7)
    for shape in [(5,), (2, 3), (4, 1)]:
        x = t.randn(shape)
        grad_out = t.randn(shape)
        g_x = chain_double_negate(grad_out, x)
        # two negations cancel -> leaf grad equals grad_out exactly
        assert t.equal(g_x, grad_out)
    # autograd cross-check on y = -(-x)
    xa = t.randn(5, requires_grad=True)
    ((-(-xa)).sum()).backward()
    manual = chain_double_negate(t.ones(5), xa.detach())
    assert t.equal(manual, xa.grad)


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(4)

def negative_back(grad_out, out, x):
    return -grad_out

def chain_double_negate(grad_out, x_leaf):
    u = -x_leaf
    y = -u
    g_u = negative_back(grad_out, y, u)
    g_x = negative_back(g_u, u, x_leaf)
    return g_x

x = t.randn(3)
grad_out = t.randn(3)
print(chain_double_negate(grad_out, x))
```
</details>